# Basic Input/Output

**Topic:** `1.7` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Render an aligned table

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** f-strings, format specs, loops

**Explanation and reasoning**

Each line is one f-string whose format spec states the alignment and width explicitly, so no manual padding is needed. Collecting the lines and joining once avoids the quadratic cost of repeated concatenation, and the header is emitted unconditionally so an empty table still has its labels.

In [ ]:
def format_table(rows) -> str:
    lines = [f"{'ROBOT':<10}{'BATTERY':>8}", '-' * 18]
    for name, pct in rows:
        lines.append(f'{name:<10}{pct:>8.1f}')
    return '\n'.join(lines)

**Complexity**

Time O(n) in the number of rows; space O(n).

**Edge cases and failure modes**

An empty list returns just the header and separator.

**Alternative approaches**

Column widths could be computed from the data to right-size the table.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = format_table([('rover-01', 48.0)])
assert 'ROBOT' in out and 'rover-01' in out
assert format_table([]).startswith('ROBOT')

---

## Solution 2 — Parse a percentage from text

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, conversion, error handling

**Explanation and reasoning**

The type guard comes first so a float argument fails loudly rather than being silently stringified. Stripping the percent sign and then re-stripping handles ' 48.5 % ' where a single strip would leave a space that float() would then reject.

In [ ]:
def parse_percent(text, default=None):
    if isinstance(text, bool) or not isinstance(text, str):
        raise TypeError(f'expected a string, got {type(text).__name__}')
    cleaned = text.strip()
    if cleaned.endswith('%'):
        cleaned = cleaned[:-1].strip()
    try:
        return float(cleaned)
    except ValueError:
        return default

**Complexity**

Time O(n) in the text length; space O(n).

**Edge cases and failure modes**

'n/a' returns the default; '48.5' without a sign parses normally.

**Alternative approaches**

Using str.removesuffix(' %') handles the spaced variant in one call.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert parse_percent('48.5%') == 48.5
assert parse_percent(' 48.5 % ') == 48.5
assert parse_percent('n/a', default=0.0) == 0.0

---

## Solution 3 — Echo lines from a stream

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** streams, loops, strings

**Explanation and reasoning**

The explicit read-then-check loop is clearer than a walrus condition here, because the end-of-input test is the interesting part and deserves its own line. A blank line yields an empty entry rather than terminating, so a file with intentional blank lines round-trips faithfully.

In [ ]:
def echo_lines(stream, prefix: str = '> ') -> list[str]:
    echoed: list[str] = []
    while True:
        line = stream.readline()
        if not line:
            break
        echoed.append(prefix + line.strip())
    return echoed

**Complexity**

Time O(n) in the stream; space O(n).

**Edge cases and failure modes**

A final line without a trailing newline is still returned.

**Alternative approaches**

Iterating the stream directly avoids repeated readline calls.

**Tests — run the cell to verify the reference solution**

In [ ]:
import io
assert echo_lines(io.StringIO('a\nb\n')) == ['> a', '> b']
assert echo_lines(io.StringIO('')) == []
assert echo_lines(io.StringIO('a\n\nb\n')) == ['> a', '> ', '> b']

---

## Solution 4 — Centre a title banner

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, format specs

**Explanation and reasoning**

The length guard returns the title untouched rather than truncating it, because a banner that silently drops part of a robot's name is worse than one that overflows its column. centre handles the padding, including the uneven left-right split for odd differences.

In [ ]:
def banner(title: str, width: int = 24, fill: str = '=') -> str:
    if width < 3:
        raise ValueError('width must be at least 3')
    if len(title) >= width:
        return title
    return title.center(width, fill)

**Complexity**

Time O(n) in the width; space O(n).

**Edge cases and failure modes**

A title exactly as long as the width is returned unchanged.

**Alternative approaches**

A format spec of ^<width also centres and never truncates.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert banner('ROBO-X') == '=========ROBO-X========='
assert len(banner('x', 7)) == 7
assert banner('a-very-long-title', 8) == 'a-very-long-title'

---

## Solution 5 — Normalise a yes/no answer

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** dicts, strings, validation

**Explanation and reasoning**

Normalising with strip().lower() before the lookup means aliases are written once in the table rather than in every comparison. dict.get returns None for an unrecognised answer, which lets the caller ask again instead of crashing on a typo.

In [ ]:
CHOICES = {
    'y': 'ARM',
    'yes': 'ARM',
    '1': 'ARM',
    'true': 'ARM',
    'n': 'DISARM',
    'no': 'DISARM',
    '0': 'DISARM',
    'false': 'DISARM',
}


def normalise_choice(text: str):
    if not isinstance(text, str):
        return None
    return CHOICES.get(text.strip().lower())

**Complexity**

Time O(n) in the text length; space O(1).

**Edge cases and failure modes**

An empty string returns None rather than raising.

**Alternative approaches**

A set of affirmative answers is enough if only yes/no is needed.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert normalise_choice(' Yes ') == 'ARM'
assert normalise_choice('maybe') is None
assert normalise_choice('') is None

---

## Solution 6 — Process commands from a stream

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** streams, loops, dicts

**Explanation and reasoning**

End of input is detected by the empty string readline returns, which is distinct from a blank line and correctly terminates the loop. Unknown commands write to the error stream and continue, so a typo never discards the decisions already made.

In [ ]:
import io

COMMANDS = {'1': 'ARM', '2': 'DISARM', 'q': 'QUIT'}


def run_commands(stream, out, errors) -> list[str]:
    history: list[str] = []
    while True:
        line = stream.readline()
        if not line:
            break
        choice = line.strip().lower()
        action = COMMANDS.get(choice)
        if action is None:
            print(f'unknown command: {choice!r}', file=errors)
            continue
        if action == 'QUIT':
            break
        history.append(action)
        print(action, file=out)
    return history

**Complexity**

Time O(n) in the commands; space O(n) in the history.

**Edge cases and failure modes**

A stream with no trailing newline still terminates cleanly.

**Alternative approaches**

Iterating the stream directly is shorter and equally correct.

**Tests — run the cell to verify the reference solution**

In [ ]:
out, err = io.StringIO(), io.StringIO()
assert run_commands(io.StringIO('1\nq\n'), out, err) == ['ARM']
assert run_commands(io.StringIO(''), out, err) == []

---

## Solution 7 — Capture what a function printed

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** contextlib, io, testing

**Explanation and reasoning**

The context manager restores the original stream in a finally block, so an exception inside func leaves sys.stdout exactly as it was - which is the reason to prefer it over a manual save-and-restore. Taking *args and **kwargs lets the helper wrap any function, not just zero-argument ones.

In [ ]:
import io
from contextlib import redirect_stdout


def capture_output(func, *args, **kwargs) -> str:
    """Run func with stdout redirected and return everything it printed."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        func(*args, **kwargs)
    return buffer.getvalue()

**Complexity**

Time and space proportional to the output produced.

**Edge cases and failure modes**

A function that prints nothing returns an empty string.

**Alternative approaches**

capsys in pytest does the same thing for test functions.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert capture_output(lambda: print('hi')) == 'hi\n'
assert capture_output(lambda: None) == ''

---

## Solution 8 — Emit a fleet report on two streams

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** sys.stderr, f-strings, iteration

**Explanation and reasoning**

The data line and the warning go to different streams in separate statements, so a caller piping stdout to a file never captures a warning as if it were data. Continuing after a missing reading keeps one incomplete record from suppressing the rest of the fleet.

In [ ]:
import sys


def report_fleet(robots, out=sys.stdout, errors=sys.stderr) -> None:
    for robot in robots:
        name = robot.get('name', '<unnamed>')
        pct = robot.get('battery_pct')
        if pct is None:
            print(f'warning: {name} has no battery reading', file=errors)
            continue
        print(f'{name}:{pct:.1f}', file=out)
        if pct < 20:
            print(f'warning: {name} battery {pct:.1f}%', file=errors)

**Complexity**

Time O(n); space O(1).

**Edge cases and failure modes**

A robot with no name is reported as '<unnamed>' rather than raising.

**Alternative approaches**

A generator of lines would be more testable but defers the stream decision.

**Tests — run the cell to verify the reference solution**

In [ ]:
import io
out, err = io.StringIO(), io.StringIO()
report_fleet([{'name': 'r1', 'battery_pct': 12.0}], out, err)
assert out.getvalue() == 'r1:12.0\n'
assert 'low' in err.getvalue() or '12.0' in err.getvalue()

---

## Solution 9 — Parse simple command-line arguments

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, sys.argv, validation

**Explanation and reasoning**

Advancing the index by two after consuming a value is the part that is easy to get wrong with a naive increment; the continue keeps the two paths explicit. Validating the count separately from the conversion means a non-numeric and a negative value each get their own message.

In [ ]:
import sys

DEFAULTS = {'name': 'rover-01', 'count': 1, 'verbose': False}


def parse_args(args) -> dict:
    options = dict(DEFAULTS)
    index = 0
    while index < len(args):
        token = args[index]
        if token == '--verbose':
            options['verbose'] = True
        elif token in {'--name', '--count'}:
            if index + 1 >= len(args):
                sys.exit(f'{token} requires a value')
            value = args[index + 1]
            if token == '--name':
                options['name'] = value
            else:
                try:
                    count = int(value)
                except ValueError:
                    sys.exit(f'--count must be an integer, got {value!r}')
                if count < 0:
                    sys.exit('--count must not be negative')
                options['count'] = count
            index += 2
            continue
        else:
            sys.exit(f'unknown argument: {token!r}')
        index += 1
    return options

**Complexity**

Time O(n) in the number of arguments; space O(1).

**Edge cases and failure modes**

Repeating a flag is harmless; the last value wins for valued options.

**Alternative approaches**

argparse handles all of this, and is the right tool in Module 9.

**Tests — run the cell to verify the reference solution**

In [ ]:
options = parse_args(['--name', 'rover-01'])
assert options == {'name': 'rover-01', 'count': 1, 'verbose': False}
assert parse_args([]) == DEFAULTS

---

## Solution 10 — Build a testable command-line entry point

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** sys.argv, streams, composition

**Explanation and reasoning**

Taking argv as a parameter is what makes this testable: a test supplies a list instead of mutating sys.argv. Returning distinct codes for usage errors and unknown commands lets a shell distinguish 'you called me wrong' from 'I do not know that'.

In [ ]:
import sys

EXIT_OK, EXIT_UNKNOWN, EXIT_USAGE = 0, 1, 2


def main(argv, out=sys.stdout, errors=sys.stderr) -> int:
    if not argv:
        print('usage: roboctl <command>', file=errors)
        return EXIT_USAGE
    command = argv[0]
    if command == 'status':
        print('STATUS ACTIVE', file=out)
        return EXIT_OK
    if command == 'stop':
        print('STOPPED', file=out)
        return EXIT_OK
    print(f'unknown command: {command!r}', file=errors)
    return EXIT_UNKNOWN

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

An empty argv returns the usage code without touching stdout.

**Alternative approaches**

argparse would also set the exit code, at the cost of a global.

**Tests — run the cell to verify the reference solution**

In [ ]:
import io
out, err = io.StringIO(), io.StringIO()
assert main(['status'], out, err) == 0
assert main([], out, err) == 2
assert main(['nope'], out, err) == 1

---

# Robotics challenge solution — ROBO-X Challenge: Dispatch Console

**Explanation**

Separating rendering from printing means rover_lines can be asserted on without capturing any output at all, while dispatch owns the stream and exit-code decisions. The command never reads sys.argv itself, so the whole entry point is a pure function of its arguments.

In [ ]:
import io
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402


def rover_lines(robot) -> list[str]:
    """Render a machine-readable line per robot, ready to print."""
    state = robot.status()
    return [f"{state['name']}:{state['battery_pct']:.1f}"]


def dispatch(argv, out, errors) -> int:
    """Print the requested view; return an exit code. No globals touched."""
    if not argv:
        print('usage: roverctl status', file=errors)
        return 2
    if argv[0] != 'status':
        print(f"unknown command: {argv[0]!r}", file=errors)
        return 1
    robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
    for line in rover_lines(robot):
        print(line, file=out)
    return 0

**Complexity**

Time O(1); space O(1).

**Notes and trade-offs**

A missing argument returns 2 without producing output.

In [ ]:
out, err = io.StringIO(), io.StringIO()
assert dispatch(['status'], out, err) == 0
assert 'rover-01' in out.getvalue()
assert dispatch([], out, err) == 2

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.